# 🧱 **Module 2: Delta Table Configuration & Design**
## "Fix the LEGO Lakehouse"

**Duration:** 45 minutes | **Level:** 300–400

---

### Scenario

The LEGO manufacturing analytics team ran their initial data pipeline with **every Spark and Delta optimization disabled** — no auto-compaction, no optimize-write, no adaptive query execution, no V-Order, no deletion vectors. The result? Thousands of tiny files, full table scans on every query, and painfully slow DML operations.

**Your mission:** Fix each table live, one optimization at a time, and measure the impact.

### Lab Pattern

Every exercise follows the same steps:

| Step | What you do |
|------|------------|
| 🐌 **Benchmark** | Run a query and capture the baseline time/metric |
| 🔍 **Diagnose** | Inspect table metadata to prove the root cause |
| 🔧 **Fix** | Apply the optimization |
| 🚀 **Re-benchmark** | Run the same test and compare against the baseline |

### Exercises

| # | Optimization | Table | What's broken |
|---|-------------|-------|----------------|
| 1 | `OPTIMIZE` (compaction) | `manufacturing_event` | Thousands of tiny files from streaming |
| 2 | Optimize Write | `inventory_transaction` | Every commit adds many small files instead of one |
| 3 | Liquid Clustering | `inventory_transaction` | Selective queries scan every file |
| 4 | Deletion Vectors | `web_order_line` | DELETE/UPDATE rewrites entire files |
| 5 | Data Skipping Stats | `wide_order_analysis` | Stats only cover first 32 cols — filter column is invisible |

---


In [58]:
%run _benchmark_utils

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 60, Finished, Available, Finished, True)

In [6]:
# ============================================================
# SETUP — Helpers
# ============================================================
import time
from pyspark.sql import DataFrame, functions as F
from delta.tables import DeltaTable

ORIG_SCHEMA = "bronze"        # original unoptimized tables (never modified)
FAST_SCHEMA = "bronze_fast"       # shallow clones we'll fix during the lab

if "_BenchmarkProxy" not in globals():
    raise NotImplementedError("_benchmark_utils was not run! Run the prior cell!")

print("\u2705 Setup complete — helpers loaded")
print(f"   Schemas: orig={ORIG_SCHEMA}, lab={FAST_SCHEMA}")


StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 8, Finished, Available, Finished, False)

✅ Setup complete — helpers loaded
   Schemas: orig=bronze_OLD2, lab=bronze_fast


In [11]:
# ============================================================
# SETUP — Shallow clone tables into lab schema
# ============================================================
# We clone the unoptimized tables so the originals stay untouched.
# This lets you re-run the lab without re-seeding data.
# Shallow clones reference the same Parquet files — instant, zero copy cost.

LAB_TABLES = ['colors', 'customer', 'inventories', 'inventory_parts', 'inventory_sets', 'inventory_transaction', 'manufacturing_event', 'mold', 'part_categories', 'parts', 'product_return', 'production_line', 'production_order', 'quality_inspection', 'set_price_history', 'sets', 'themes', 'web_order', 'web_order_line']

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FAST_SCHEMA}")

for table in LAB_TABLES:
    spark.sql(f"DROP TABLE IF EXISTS {FAST_SCHEMA}.{table}")
    spark.sql(f"""
        CREATE TABLE {FAST_SCHEMA}.{table}
        SHALLOW CLONE {ORIG_SCHEMA}.{table}
    """)
    print(f"  ✅ {FAST_SCHEMA}.{table} ← shallow clone of {ORIG_SCHEMA}.{table}")

# All exercises operate on FAST_SCHEMA
print(f"\n📌 All exercises will use schema: {FAST_SCHEMA}")
print(f"   Original tables in '{ORIG_SCHEMA}' are preserved for re-runs.")


StatementMeta(, ff579f3e-cf06-42b4-b027-649d5c37702a, 13, Finished, Available, Finished, False)

  ✅ bronze_fast.colors ← shallow clone of bronze_OLD2.colors
  ✅ bronze_fast.customer ← shallow clone of bronze_OLD2.customer
  ✅ bronze_fast.inventories ← shallow clone of bronze_OLD2.inventories
  ✅ bronze_fast.inventory_parts ← shallow clone of bronze_OLD2.inventory_parts
  ✅ bronze_fast.inventory_sets ← shallow clone of bronze_OLD2.inventory_sets
  ✅ bronze_fast.inventory_transaction ← shallow clone of bronze_OLD2.inventory_transaction
  ✅ bronze_fast.manufacturing_event ← shallow clone of bronze_OLD2.manufacturing_event
  ✅ bronze_fast.mold ← shallow clone of bronze_OLD2.mold
  ✅ bronze_fast.part_categories ← shallow clone of bronze_OLD2.part_categories
  ✅ bronze_fast.parts ← shallow clone of bronze_OLD2.parts
  ✅ bronze_fast.product_return ← shallow clone of bronze_OLD2.product_return
  ✅ bronze_fast.production_line ← shallow clone of bronze_OLD2.production_line
  ✅ bronze_fast.production_order ← shallow clone of bronze_OLD2.production_order
  ✅ bronze_fast.quality_inspection ← 

### 🔍 Before We Tune: Let's See the Data

Before diving into configuration, let's peek at what we're working with. This is **real LEGO catalog data** — actual sets and themes that are combined with synthetic manufacturing and sales events.

In [2]:
# What LEGO sets are people ordering?
print("🧱 Top 10 Most-Ordered LEGO Sets\n")
display(spark.sql(f"""
    SELECT s.name AS set_name, s.set_num, t.name AS theme, s.year, s.num_parts,
           COUNT(*) AS times_ordered, ROUND(SUM(wol.extended_price), 2) AS total_revenue
    FROM {ORIG_SCHEMA}.web_order_line wol
    JOIN {ORIG_SCHEMA}.sets s ON wol.set_num = s.set_num
    LEFT JOIN {ORIG_SCHEMA}.themes t ON s.theme_id = t.id
    GROUP BY s.name, s.set_num, t.name, s.year, s.num_parts
    ORDER BY times_ordered DESC
    LIMIT 10
"""))

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 4, Finished, Available, Finished, False)

🧱 Top 10 Most-Ordered LEGO Sets

Calling getAccessToken from PyTridentTokenLibrary


SynapseWidget(Synapse.DataFrame, 87fcd250-eb78-430f-84c9-eac5f9ccc530)

In [3]:
# What's happening on the factory floor?
print("\n🏭 Manufacturing Defect Breakdown\n")
display(spark.sql(f"""
    SELECT defect_type, COUNT(*) AS defect_count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM {ORIG_SCHEMA}.manufacturing_event
    WHERE defect_detected = true
    GROUP BY defect_type
    ORDER BY defect_count DESC
"""))

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 5, Finished, Available, Finished, False)


🏭 Manufacturing Defect Breakdown



SynapseWidget(Synapse.DataFrame, 34c77d23-fedc-4e6a-9d7c-e664d12fbafc)

---

# Warm-up: The Cost of Schema Inference

Before we even run queries, notice how long it takes to simply **discover the schema** from the landing zone. With thousands of small files, Spark must open many of them to infer column types — this is a hidden startup cost every time the pipeline restarts.

**Production best practice:** Define schemas statically so your pipeline starts instantly — no file scanning needed.

---

In [65]:
# ============================================================
# WARM-UP — Time schema inference on the landing zone
# ============================================================

# The unoptimized pipeline left thousands of small files in the landing zone.
# Schema inference must scan these files to discover the schema.
#
# NOTE: spark.read.json() triggers inference EAGERLY at creation time,
# so we must time the DataFrame construction itself — not an action.

LANDING_TABLE = "manufacturing_event"
landing_path = f"Files/landing/{LANDING_TABLE}"

print(f"🐌 Inferring schema from landing zone: {landing_path}\n")
print("   Spark must open files and read metadata to discover columns + types...\n")

with benchmark_op("Schema Inference", "inferred (file scan)", spark):
    inferred_df = spark.read.option("multiline", "true").json(landing_path)

inferred_df.printSchema()

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 67, Finished, Available, Finished, False)

🐌 Inferring schema from landing zone: Files/landing/manufacturing_event

   Spark must open files and read metadata to discover columns + types...

⏱️  Schema Inference [inferred (file scan)]: 26539.77 ms
root
 |-- _corrupt_record: string (nullable = true)



In [66]:
# ============================================================
# WARM-UP — Compare: static schema definition (instant)
# ============================================================
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, BooleanType, DecimalType

# A production pipeline defines the schema once in code — no file scanning needed
static_schema = StructType([
    StructField("EventId", StringType()),
    StructField("Timestamp", TimestampType()),
    StructField("MachineId", StringType()),
    StructField("PartNum", StringType()),
    StructField("ColorId", IntegerType()),
    StructField("MoldTemp", DecimalType(5, 1)),
    StructField("InjectionPressure", DecimalType(6, 1)),
    StructField("CycleTimeMs", IntegerType()),
    StructField("DefectDetected", BooleanType()),
    StructField("DefectType", StringType()),
    StructField("BatchId", StringType()),
])

with benchmark_op("Schema Inference", "static (no scan)", spark):
    static_df = spark.read.schema(static_schema).json(landing_path)

static_df.printSchema()

print("\n📝 Takeaway: production pipelines define schemas upfront.")
print("   Inference is convenient for exploration but adds startup latency,")
print("   especially when scanning thousands of small files.")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 68, Finished, Available, Finished, False)

⏱️  Schema Inference [static (no scan)]: 966.29 ms

  ┌──────────────────────────────────────────────────────────┐
  │  Schema Inference                                        │
  ├──────────────────────────────────────────────────────────┤
  │  State                          Time (ms)        Factor  │
  ├──────────────────────────────────────────────────────────┤
  │  inferred (file scan)            26539.77      baseline  │
  │  static (no scan)                  966.29  27.5x faster  │
  └──────────────────────────────────────────────────────────┘
root
 |-- EventId: string (nullable = true)
 |-- Timestamp: timestamp (nullable = true)
 |-- MachineId: string (nullable = true)
 |-- PartNum: string (nullable = true)
 |-- ColorId: integer (nullable = true)
 |-- MoldTemp: decimal(5,1) (nullable = true)
 |-- InjectionPressure: decimal(6,1) (nullable = true)
 |-- CycleTimeMs: integer (nullable = true)
 |-- DefectDetected: boolean (nullable = true)
 |-- DefectType: string (nullable = true)
 |

---

# Exercise 1: Fix the Small Files Problem

**Table:** `manufacturing_event` — high-frequency IoT telemetry from the injection molding floor

**What's wrong:** The unoptimized pipeline wrote data via Spark Structured Streaming with no auto-compaction and no optimize-write. Every micro-batch created a new tiny Parquet file. The result: thousands of files, each just a few KB.

**Why it matters:**
- Each file requires a separate Spark task → scheduling overhead dominates actual compute
- File metadata (Parquet footer, Delta stats) is disproportionately large vs. data
- The Delta transaction log bloats with thousands of `AddFile` entries

**Fix:** [`OPTIMIZE`](https://learn.microsoft.com/fabric/data-engineering/delta-optimization-and-v-order?tabs=sparksql#optimize) — compacts small files into ~128 MB target files.

---

In [18]:
# ============================================================
# 1️⃣ BENCHMARK — Capture baseline query time
# ============================================================

print("🐌 Running baseline query on manufacturing_event...\n")

with benchmark_op("OPTIMIZE (compaction)", "before", spark):
    part_query = spark.sql(f"SELECT part_num, SUM(cycle_time_ms) AS total_cycle_time FROM {FAST_SCHEMA}.manufacturing_event GROUP BY part_num").collect()

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 20, Finished, Available, Finished, False)

🐌 Running baseline query on manufacturing_event...

⏱️  OPTIMIZE (compaction) [before]: 1834.24 ms

  ┌──────────────────────────────────────────────────────────┐
  │  OPTIMIZE (compaction)                                   │
  ├──────────────────────────────────────────────────────────┤
  │  State                          Time (ms)        Factor  │
  ├──────────────────────────────────────────────────────────┤
  │  before                           1834.24      baseline  │
  │  after                             647.05   2.8x faster  │
  └──────────────────────────────────────────────────────────┘


In [8]:
# ============================================================
# 1️⃣ DIAGNOSE — Prove the root cause is small files
# ============================================================

print("🔍 Table diagnostics:\n")
metrics_1_before = show_metrics(f"{FAST_SCHEMA}.manufacturing_event", "before")

if metrics_1_before["avg_file_kb"] < 1024:
    ratio = round(131072 / max(metrics_1_before["avg_file_kb"], 1))
    print(f"\n   ⚠️  Average file is {metrics_1_before['avg_file_kb']:.0f} KB — optimal target is ~128 MB (131,072 KB)")
    print(f"   ⚠️  Files are {ratio:,}× smaller than they should be")

print("\n📋 DESCRIBE DETAIL:")
display(spark.sql(f"DESCRIBE DETAIL {FAST_SCHEMA}.manufacturing_event").select("format", "numFiles", "sizeInBytes"))

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 10, Finished, Available, Finished, False)

🔍 Table diagnostics:

   bronze_fast.manufacturing_event (before):     452 files  |     292.5 MB  |  avg    662.6 KB/file

   ⚠️  Average file is 663 KB — optimal target is ~128 MB (131,072 KB)
   ⚠️  Files are 198× smaller than they should be

📋 DESCRIBE DETAIL:


SynapseWidget(Synapse.DataFrame, d60e7c8e-92f0-4996-a163-e7579d0b2fba)

### 🎯 Challenge: Check the table file count and average file size

You've seen the problem before, thousands of tiny files that makes performance regress. Can you confirm it?

**Your task:** check the table file count and average file size of `manufacturing_event` in the `{FAST_SCHEMA}`.

> 💡 Hint: Describe the details of the table...

Try it in the cell below!

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 15 fields>

<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal</summary>

<br/>

```python
df = spark.sql(f"DESCRIBE DETAIL {FAST_SCHEMA}.manufacturing_event")
display(df)
```

**What to check:**
- sizeInBytes: the aggregate size of all files in the current snapshot, in bytes.
- numFiles: the count of files in the current snapshot.

Files are considered _minimally healthy_ in size if they exceed 64 MB.

</details>

---

In [13]:
# ============================================================
# 1️⃣ FIX — Run OPTIMIZE to compact small files
# ============================================================

print("🔧 Running OPTIMIZE {FAST_SCHEMA}.manufacturing_event...\n")
start = time.time()
result = spark.sql(f"OPTIMIZE {FAST_SCHEMA}.manufacturing_event")
print(f"   Completed in {round(time.time() - start, 1)}s")
display(result)

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 15, Finished, Available, Finished, False)

🔧 Running OPTIMIZE {FAST_SCHEMA}.manufacturing_event...

   Completed in 11.9s


SynapseWidget(Synapse.DataFrame, d8d724fc-224d-4d80-b8e3-72638821f304)

In [17]:
# ============================================================
# 1️⃣ RE-BENCHMARK — Same query, compare against baseline
# ============================================================

print("🚀 Running same query after OPTIMIZE...\n")

with benchmark_op("OPTIMIZE (compaction)", "after", spark):
    part_query_fast = spark.sql(f"SELECT part_num, SUM(cycle_time_ms) AS total_cycle_time FROM {FAST_SCHEMA}.manufacturing_event GROUP BY part_num").collect()

metrics_1_after = show_metrics(f"{FAST_SCHEMA}.manufacturing_event", "after")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 19, Finished, Available, Finished, False)

🚀 Running same query after OPTIMIZE...

⏱️  OPTIMIZE (compaction) [after]: 647.05 ms

  ┌──────────────────────────────────────────────────────────┐
  │  OPTIMIZE (compaction)                                   │
  ├──────────────────────────────────────────────────────────┤
  │  State                          Time (ms)        Factor  │
  ├──────────────────────────────────────────────────────────┤
  │  before                           2263.66      baseline  │
  │  after                             647.05   3.5x faster  │
  └──────────────────────────────────────────────────────────┘
   bronze_fast.manufacturing_event (after):      28 files  |     252.8 MB  |  avg   9244.0 KB/file


### 💡 What Just Happened?

`OPTIMIZE` rewrote all those tiny files into a smaller number of properly-sized files (~128 MB each). This is a **one-time reactive compaction** — it doesn't prevent small files from appearing again on the next write.

> 📝 **Note:** `OPTIMIZE` creates new files but keeps old files for time travel. Run `VACUUM` to reclaim storage (default retention: 7 days).

---

In [19]:
# 🧱 Let's see what optimized manufacturing data looks like!
print("🏭 Top 10 Busiest Machines (by event count)\n")
display(spark.sql(f"""
    SELECT machine_id,
           COUNT(*) AS total_events,
           SUM(CASE WHEN defect_detected THEN 1 ELSE 0 END) AS defects,
           ROUND(AVG(mold_temp), 1) AS avg_mold_temp,
           ROUND(AVG(cycle_time_ms), 0) AS avg_cycle_ms
    FROM {FAST_SCHEMA}.manufacturing_event
    GROUP BY machine_id
    ORDER BY total_events DESC
    LIMIT 10
"""))

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 21, Finished, Available, Finished, False)

🏭 Top 10 Busiest Machines (by event count)



SynapseWidget(Synapse.DataFrame, 2c1715cf-bfae-4e71-b4e6-d335be45eb8b)

---

# Exercise 2: Optimize Write

**Table:** `inventory_transaction` — part movements across production lines

**What's wrong:** Look at the Delta transaction log: every streaming commit added many small files when the data could have fit in one. Without **optimize write** (bin-packing at write time), each micro-batch creates a file-per-partition, regardless of how small the data is.

**Why it matters:**
- `OPTIMIZE` (Exercise 1) is **reactive** — you run it *after* the damage
- Optimize write is **proactive** — it bin-packs data *during* the write
- Without it, you're in an endless cycle: write small files → run OPTIMIZE → write small files again

**Approach:**
1. Analyze the Delta log to see the file-per-commit pattern from the original pipeline
2. Replicate the problem with a test write
3. Enable optimize write and repeat — see the difference

---

In [30]:
# ============================================================
# 2️⃣ DIAGNOSE — Analyze the Delta log: files added over commits
# ============================================================
from pyspark.sql.window import Window
print("🔍 Analyzing Delta transaction log for inventory_transaction...\n")

history = spark.sql(f"DESCRIBE HISTORY {ORIG_SCHEMA}.inventory_transaction")

w = Window.orderBy("version").rowsBetween(Window.unboundedPreceding, Window.currentRow)

commit_stats = (
    history
    .select(
        F.col("version"),
        F.col("timestamp"),
        F.col("operation"),
        F.col("operationMetrics.numOutputRows").cast("long").alias("rows_written"),
        F.coalesce(
            F.col("operationMetrics.numFiles").cast("int"),
            F.col("operationMetrics.numAddedFiles").cast("int")
        ).alias("files_added"),
        F.lit(1).alias("files_if_ow_enabled")
    )
    .filter("files_added IS NOT NULL AND files_added > 0")
    .withColumn("iteration", F.row_number().over(Window.orderBy("version")))
    .withColumn("compaction_group", F.floor((F.col("iteration") - 1) / 50))
    .orderBy("version")
)

w_compaction_group = (
    Window
    .partitionBy("compaction_group")
    .orderBy("version")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

commit_stats_agg = (
    commit_stats
    .withColumn(
        "cumulative_files",
        F.sum("files_added").over(w)
    )
    .withColumn(
        "simulated_files_if_ow_enabled",
        F.sum("files_if_ow_enabled").over(w)
    )
    .withColumn(
        "simulated_files_if_ow_and_ac_enabled",
        F.sum("files_if_ow_enabled").over(w_compaction_group)
    )
    .withColumn(
        "cuml_rows_written",
        F.sum(F.coalesce("rows_written", F.lit(0))).over(w)
    )
)

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 32, Finished, Available, Finished, False)

🔍 Analyzing Delta transaction log for inventory_transaction...



### 🎯 Visualize file growth over time
Chart the cumulative files added over time vs. the cumulative files added if Optimize Write were enabled to bin-pack small partitions of data before writing.
1. Select **New chart**
1. Create a **Line Chart**
1. Add **version** to the _X-axis_
1. Add the following fields to the _Y-axis_:
    - **cumulative_files_added**
    - **simulated_files_if_ow_enabled**
    - **simulated_files_if_ow_and_ac_enabled**

Without running `OPTIMIZE`, this table will have 2-3x the number of files causing unnecessary slowness for queries and update operations.

In [29]:
display(commit_stats_agg)

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b4dc05dd-840c-488c-8a44-95f6a89659d7)

In [36]:
# ============================================================
# 2️⃣ BENCHMARK — Replicate: write WITHOUT optimize write
# ============================================================

# Start from a clean compacted state
spark.sql(f"OPTIMIZE {FAST_SCHEMA}.inventory_transaction")
files_before = get_table_metrics(f"{FAST_SCHEMA}.manufacturing_event")["num_files"]

# Simulate a streaming micro-batch by appending data
# Use repartition to mimic how streaming creates multiple partitions
BATCH_ROWS = 5000

print(f"🐌 Appending {BATCH_ROWS:,} rows WITHOUT optimize write (repartitioned to 8 files)...\n")

batch = spark.table(f"{FAST_SCHEMA}.manufacturing_event").limit(BATCH_ROWS).repartition(8)
batch.write.format("delta").mode("append").saveAsTable(f"{FAST_SCHEMA}.manufacturing_event")

files_after_bad = get_table_metrics(f"{FAST_SCHEMA}.manufacturing_event")["num_files"]
new_files_without_ow = files_after_bad - files_before
print(f"   Files before append: {files_before:,}")
print(f"   Files after append:  {files_after_bad:,}")
print(f"   ⚠️  New files created: {new_files_without_ow}")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 38, Finished, Available, Finished, False)

🐌 Appending 5,000 rows WITHOUT optimize write (repartitioned to 8 files)...

   Files before append: 28
   Files after append:  29
   ⚠️  New files created: 1


### 🎯 Challenge: Mitigate Future Small Files

You've fixed the existing small files with OPTIMIZE. But new writes will create small files again unless you change the table's write behavior.

**Your task:** Set the `delta.autoOptimize.optimizeWrite` and `delta.autoOptimize.autoCompact` table properties to `'true'` on `manufacturing_event`.

> 💡 Hint: Use `ALTER TABLE ... SET TBLPROPERTIES`

In [37]:
# YOUR CODE HERE
# Enable optimize write on manufacturing_event
# spark.sql(f"ALTER TABLE ...")


StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 39, Finished, Available, Finished, False)

DataFrame[]

<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal</summary>

<br/>

```python
spark.sql(f"""
    ALTER TABLE {FAST_SCHEMA}.manufacturing_event
    SET TBLPROPERTIES (
      'delta.autoOptimize.optimizeWrite' = 'true',
      'delta.autoOptimize.autoCompact' = 'true'
    )
""")
```

**Optimize Write vs Auto Compact:**
- **Optimize Write** — coalesces small partitions at write time (prevents small files)
- **Auto Compact** — runs a mini-OPTIMIZE after each write (fixes small files after the fact)
- Best practice: enable both for most tables

</details>

---

In [38]:
# =======================================================================
# 2️⃣ RE-BENCHMARK — Same append, now with optimize write set on the table
# =======================================================================

# Clean baseline
spark.sql(f"OPTIMIZE {FAST_SCHEMA}.manufacturing_event")
files_before_good = get_table_metrics(f"{FAST_SCHEMA}.manufacturing_event")["num_files"]

print(f"🚀 Appending {BATCH_ROWS:,} rows WITH optimize write (same repartition to 8)...\n")

batch = spark.table(f"{FAST_SCHEMA}.manufacturing_event").limit(BATCH_ROWS).repartition(8)
batch.write.format("delta").mode("append").saveAsTable(f"{FAST_SCHEMA}.manufacturing_event")

files_after_good = get_table_metrics(f"{FAST_SCHEMA}.manufacturing_event")["num_files"]
new_files_with_ow = files_after_good - files_before_good

print(f"   Files before append: {files_before_good:,}")
print(f"   Files after append:  {files_after_good:,}")
print(f"   New files created:   {new_files_with_ow}")

print(f"\n{'=' * 60}")
print(f"  Exercise 2: Optimize Write")
print(f"{'=' * 60}")
print(f"  {'Metric':<30} {'Without OW':>12} {'With OW':>12}")
print(f"  {'-' * 54}")
print(f"  {'New files per append':<30} {new_files_without_ow:>12} {new_files_with_ow:>12}")
print(f"  {'Overhead vs. ideal (1 file)':<30} {new_files_without_ow:>11}x {'1x':>12}")
print(f"{'=' * 60}")

benchmarks["Exercise 2: Optimize Write"] = {
    "before_s": new_files_without_ow, "after_s": new_files_with_ow,
    "speedup": round(new_files_without_ow / max(new_files_with_ow, 1), 1),
    "before_files": new_files_without_ow, "after_files": new_files_with_ow,
    "note": "files created per append"
}

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 40, Finished, Available, Finished, False)

🚀 Appending 5,000 rows WITH optimize write (same repartition to 8)...

   Files before append: 29
   Files after append:  30
   New files created:   1

  Exercise 2: Optimize Write
  Metric                           Without OW      With OW
  ------------------------------------------------------
  New files per append                      1            1
  Overhead vs. ideal (1 file)              1x           1x


### 💡 What Just Happened?

Without optimize write, Spark wrote one file per output partition — even though the data was tiny. With optimize write enabled, Spark **bin-packs** the data at write time, coalescing small partitions into properly-sized files.

| Setting | Behavior |
|---------|----------|
| `optimizeWrite = false` | 1 file per partition → 8 files for 5,000 rows |
| `optimizeWrite = true` | Bin-packs into ~1 file (data fits in one ~128 MB target) |
| `autoCompact = true` | Triggers synchronous compaction after writes exceed small file threshold (defaults to 50) |

> 📝 **Optimize write is one of the most important settings for preventing small files.** It mitigates small file issue pre-write — no post-hoc `OPTIMIZE` needed.
---

---

# Exercise 3: Liquid Clustering

**Table:** `inventory_transaction` (continuing from Exercise 2)

**Columns:** `TransactionId`, `Timestamp`, `PartNum`, `ColorId`, `LineId`, `TransactionType`, `Quantity`, `ReferenceId`

**What's wrong:** Even after compaction, data files contain a random mix of all values. When you filter by `PartNum` or `TransactionType`, Spark must scan **every file** because the min/max file-level statistics overlap across all files — nothing can be skipped.

**Why it matters:**
- A query for one specific part number scans 100% of the data
- No files can be pruned via Delta data skipping
- Gets worse as the table grows

**Fix:** [Liquid clustering](https://learn.microsoft.com/en-us/fabric/data-engineering/liquid-clustering?tabs=sparksql) — co-locates related rows in the same files so Delta can skip irrelevant files entirely.

---

In [42]:
# ============================================================
# 3️⃣ BENCHMARK — Run a selective query (scans everything)
# ============================================================

# Compact first so file layout is clean. Use a 1m targetFileSize to ensure enough files to demonstrate file skipping 
spark.sql(f"ALTER TABLE {FAST_SCHEMA}.inventory_transaction SET TBLPROPERTIES ('delta.targetFileSize' = '1m')")
spark.sql(f"OPTIMIZE {FAST_SCHEMA}.inventory_transaction")
metrics_3_before = show_metrics(f"{FAST_SCHEMA}.inventory_transaction", "before clustering")

print(f"\n🐌 Running selective query: WHERE color_id = 8...\n")

query = spark.sql(f"""
    SELECT transaction_type, COUNT(*) AS cnt, SUM(quantity) AS total_qty
    FROM {FAST_SCHEMA}.inventory_transaction
    WHERE color_id = 8
    GROUP BY transaction_type
""")
with benchmark_op("Liquid Clustering", "before (no clustering)", spark):
    query.collect()

print(f"\n🔍 Files scanned in query: {len(query.inputFiles())}\n")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 44, Finished, Available, Finished, False)

   bronze_fast.inventory_transaction (before clustering):      46 files  |      39.3 MB  |  avg    875.3 KB/file

🐌 Running selective query: WHERE color_id = 8...

⏱️  Liquid Clustering [before (no clustering)]: 1823.16 ms

🔍 Files scanned in query: 46



In [43]:
# ============================================================
# 3️⃣ FIX — Apply liquid clustering and re-optimize
# ============================================================

print("🔧 Applying liquid clustering: CLUSTER BY (color_id)...\n")
spark.sql(f"ALTER TABLE {FAST_SCHEMA}.inventory_transaction CLUSTER BY (color_id)")
print("✅ Clustering columns set\n")

print("Running OPTIMIZE to rewrite files with clustering layout...")
start = time.time()
optimize_metrics_df = spark.sql(f"OPTIMIZE {FAST_SCHEMA}.inventory_transaction FULL")
print(f"   Completed in {round(time.time() - start, 1)}s")

metrics_3_after = show_metrics(f"{FAST_SCHEMA}.inventory_transaction", "after clustering + OPTIMIZE")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 45, Finished, Available, Finished, False)

🔧 Applying liquid clustering: CLUSTER BY (color_id)...

✅ Clustering columns set

Running OPTIMIZE to rewrite files with clustering layout...
   Completed in 9.5s
   bronze_fast.inventory_transaction (after clustering + OPTIMIZE):      39 files  |      52.4 MB  |  avg   1375.7 KB/file


### 🎯 Challenge: Check Liquid Clustering Quality Metrics

`OPTIMIZE` metrics in Fabric Spark Runtime 2.0 contains a `clusteringQuality` struct. Query the struct to see the clustering quality.

**Your task:**

1. Drill into the `optimize_metrics_df` DataFrame to inspect the `skippingEffectiveness` of the clustering column(s).

> 💡 Hint: Use `EXPLODE` and `*` expand

In [45]:
# Extract clustering quality metrics from optimize_metrics DataFrame


StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 168293c7-1709-4489-8013-af8b16ca3689)

<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal</summary>

<br/>

```python
metrics_parsed_df = optimize_metrics_df \
    .selectExpr("explode(metrics.clusteringQuality) as clustering_quality") \
    .select("clustering_quality.*")
    
display(metrics_parsed_df)
```

</details>

---

In [46]:
# ============================================================
# 3️⃣ RE-BENCHMARK — Same query, now with clustering
# ============================================================

print(f"🚀 Running same query: WHERE color_id = 8...\n")

query = spark.sql(f"""
    SELECT transaction_type, COUNT(*) AS cnt, SUM(quantity) AS total_qty
    FROM {FAST_SCHEMA}.inventory_transaction
    WHERE color_id = 8
    GROUP BY transaction_type
""")
with benchmark_op("Liquid Clustering", "after (clustered)", spark):
    query.collect()

metrics_3_after = show_metrics(f"{FAST_SCHEMA}.inventory_transaction", "after clustering")

print(f"\n🔍 Files scanned in query: {len(query.inputFiles())}\n")

print("\n💡 Check Spark UI → SQL tab → Scan node for 'number of files read' —")
print("   with healthy clustering (see skippingEffectiveness in the prior cell), most files should be skipped for selective filters.")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 48, Finished, Available, Finished, False)

🚀 Running same query: WHERE color_id = 8...

⏱️  Liquid Clustering [after (clustered)]: 580.04 ms

  ┌──────────────────────────────────────────────────────────┐
  │  Liquid Clustering                                       │
  ├──────────────────────────────────────────────────────────┤
  │  State                          Time (ms)        Factor  │
  ├──────────────────────────────────────────────────────────┤
  │  before (no clustering)           1823.16      baseline  │
  │  after (clustered)                 580.04   3.1x faster  │
  └──────────────────────────────────────────────────────────┘
   bronze_fast.inventory_transaction (after clustering):      39 files  |      52.4 MB  |  avg   1375.7 KB/file

🔍 Files scanned in query: 1


💡 Check Spark UI → SQL tab → Scan node for 'files pruned' —
   with clustering, most files should be skipped for selective filters.


### 💡 What Just Happened?

Liquid clustering **physically re-organizes data** so rows with similar `PartNum` and `TransactionType` values end up in the same files. Delta's min/max file statistics can then immediately skip files that don't contain matching values.

| Before clustering | After clustering |
|---|---|
| Every file has a random mix of all part numbers | Files contain contiguous ranges of part numbers |
| Scanning `WHERE PartNum = 'X'` reads ALL files | Same query prunes most files via data skipping |

> 🆚 **Liquid clustering vs. partitioning:** Partitioning creates directory-level splits — great for low-cardinality columns, but creates a small files nightmare with high cardinality. Liquid clustering achieves similar data skipping **without** the small files trap, and it's incremental — new data is automatically clustered when `autoCompact` is enabled.

---

---

# Exercise 4: Deletion Vectors

**Table:** `web_order_line` — order line items for LEGO set purchases

**Columns:** `OrderId`, `LineNumber`, `SetNum`, `PartNum`, `ItemName`, `Quantity`, `UnitPrice`, `ExtendedPrice`

**What's wrong:** Without deletion vectors, every `DELETE` or `UPDATE` must **rewrite the entire Parquet files** that contain affected rows. Even deleting a single row triggers a full file rewrite.

**Why it matters:**
- A `DELETE` touching 100 rows across 10 files rewrites all 10 files completely
- Write amplification is enormous — you rewrite GBs to remove KBs
- `MERGE` operations (common in ETL) suffer the same penalty

**Fix:** [Deletion vectors](https://learn.microsoft.com/fabric/data-engineering/delta-optimization-and-v-order?tabs=sparksql#deletion-vectors) — instead of rewriting files, Delta writes a small sidecar file that marks which rows are logically deleted. The original data files stay untouched.

---

In [60]:
# 🧱 Let's peek at the rows we're about to delete
print("🗑️ Sample rows that will be deleted (defect_detected = true)\n")
display(spark.sql(f"""
    SELECT set_num, COUNT(*) as cnt
    FROM {FAST_SCHEMA}.web_order_line
    WHERE set_num IS NOT NULL
    GROUP BY set_num
    ORDER BY cnt ASC
    LIMIT 2
"""))

total_count = spark.table(f"{FAST_SCHEMA}.web_order_line").count()
print(f"\n📊 2 order lines out of {total_count:,} total ({2/total_count*100:.5f}%)")
print("   Next: we'll DELETE these rows and compare with vs. without Deletion Vectors")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 62, Finished, Available, Finished, False)

🗑️ Sample rows that will be deleted (defect_detected = true)



SynapseWidget(Synapse.DataFrame, 01786ce7-f894-4a34-8c0e-2246c7214b2c)


📊 2 order lines out of 519,834 total (0.00038%)
   Next: we'll DELETE these rows and compare with vs. without Deletion Vectors


In [61]:
# ============================================================
# 4️⃣ SETUP — Compact the table and pick two test conditions
# ============================================================

print("🔧 Preparing web_order_line (OPTIMIZE to clean baseline)...\n")
spark.sql(f"OPTIMIZE {FAST_SCHEMA}.web_order_line")
spark.sql(f"ALTER TABLE {FAST_SCHEMA}.web_order_line UNSET TBLPROPERTIES ('delta.enableDeletionVectors')")
metrics_4_before = show_metrics(f"{FAST_SCHEMA}.web_order_line", "baseline")

# Pick two different SetNums for before/after DELETE comparison
set_nums = spark.sql(f"""
    SELECT set_num, COUNT(*) as cnt
    FROM {FAST_SCHEMA}.web_order_line
    WHERE set_num IS NOT NULL
    GROUP BY set_num
    ORDER BY cnt ASC
    LIMIT 2
""").collect()

set_a, count_a = set_nums[0][0], set_nums[0][1]
set_b, count_b = set_nums[1][0], set_nums[1][1]
print(f"\n   Will DELETE set_num='{set_a}' ({count_a:,} rows) without DVs")
print(f"   Will DELETE set_num='{set_b}' ({count_b:,} rows) with DVs")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 63, Finished, Available, Finished, False)

🔧 Preparing web_order_line (OPTIMIZE to clean baseline)...

   bronze_fast.web_order_line (baseline):       1 files  |       9.8 MB  |  avg   9979.6 KB/file

   Will DELETE set_num='5003150-1' (1 rows) without DVs
   Will DELETE set_num='5005366-1' (1 rows) with DVs


In [62]:
# ============================================================
# 4️⃣ BENCHMARK — DELETE without deletion vectors (rewrites files)
# ============================================================

print(f"🐌 DELETE WHERE set_num = '{set_a}' (without deletion vectors)...\n")

with benchmark_op("Deletion Vectors", "without DVs (file rewrite)", spark):
    spark.sql(f"DELETE FROM {FAST_SCHEMA}.web_order_line WHERE set_num = '{set_a}'")

# Check the commit to see how many files were rewritten
history = spark.sql(f"DESCRIBE HISTORY {FAST_SCHEMA}.web_order_line LIMIT 1").select(
    "version", "operation", "operationMetrics"
).collect()[0]
op_metrics = history["operationMetrics"]
files_rewritten_before = int(op_metrics.get("numRemovedFiles", 0))

print(f"   Files rewritten: {files_rewritten_before}")
print(f"   ⚠️  Entire files rewritten just to remove {count_a:,} rows")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 64, Finished, Available, Finished, False)

🐌 DELETE WHERE set_num = '5003150-1' (without deletion vectors)...

⏱️  Deletion Vectors [without DVs (file rewrite)]: 5257.22 ms
   Files rewritten: 1
   ⚠️  Entire files rewritten just to remove 1 rows


In [63]:
# ============================================================
# 4️⃣ FIX — Enable deletion vectors
# ============================================================

print("🔧 Enabling deletion vectors...\n")
spark.sql(f"""
    ALTER TABLE {FAST_SCHEMA}.web_order_line SET TBLPROPERTIES (
        'delta.enableDeletionVectors' = 'true'
    )
""")

props = spark.sql(f"SHOW TBLPROPERTIES {FAST_SCHEMA}.web_order_line").filter("key LIKE '%deletionVector%'")
display(props)
print("✅ Deletion vectors enabled")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 65, Finished, Available, Finished, False)

🔧 Enabling deletion vectors...



SynapseWidget(Synapse.DataFrame, fb84aabd-a297-4190-a90f-23230e2d26db)

✅ Deletion vectors enabled


In [64]:
# ============================================================
# 4️⃣ RE-BENCHMARK — DELETE with deletion vectors (no file rewrite)
# ============================================================

print(f"🚀 DELETE WHERE SetNum = '{set_b}' (with deletion vectors)...\n")

with benchmark_op("Deletion Vectors", "with DVs (sidecar only)", spark):
    spark.sql(f"DELETE FROM {FAST_SCHEMA}.web_order_line WHERE set_num = '{set_b}'")

history = spark.sql(f"DESCRIBE HISTORY {FAST_SCHEMA}.web_order_line LIMIT 1").select(
    "version", "operation", "operationMetrics"
).collect()[0]
op_metrics = history["operationMetrics"]
files_rewritten_after = int(op_metrics.get("numRemovedFiles", 0))

print(f"   Files rewritten: {files_rewritten_after}")
print(f"\n💡 With DVs, Delta wrote a tiny sidecar file marking deleted rows.")
print(f"   The original data files were NOT rewritten — massive write amplification savings.")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 66, Finished, Available, Finished, False)

🚀 DELETE WHERE SetNum = '5005366-1' (with deletion vectors)...

⏱️  Deletion Vectors [with DVs (sidecar only)]: 5525.15 ms

  ┌──────────────────────────────────────────────────────────┐
  │  Deletion Vectors                                        │
  ├──────────────────────────────────────────────────────────┤
  │  State                          Time (ms)        Factor  │
  ├──────────────────────────────────────────────────────────┤
  │  without DVs (file rewrite)       5257.22      baseline  │
  │  with DVs (sidecar only)          5525.15          1.0x  │
  └──────────────────────────────────────────────────────────┘
   Files rewritten: 0

💡 With DVs, Delta wrote a tiny sidecar file marking deleted rows.
   The original data files were NOT rewritten — massive write amplification savings.


---

# Exercise 5: Data Skipping Stats on Wide Tables

**Table:** `wide_order_analysis` — a denormalized join of orders, customers, inventory, and product data

**What’s wrong:** Delta collects min/max file statistics for only the **first 32 columns** by default (`delta.dataSkippingNumIndexedCols = 32`). When a wide table has a filter column beyond position 32, Delta has no stats for it — and will **block you from clustering** on that column with `DELTA_CLUSTERING_COLUMN_MISSING_STATS`.

**Why it matters:**
- Denormalized / wide tables are common in analytics lakehouses
- Without stats, data skipping is blind — every file is scanned
- Clustering requires stats on the clustering column — you can’t even enable it
- No warning until you try — queries silently scan everything

**Exercise flow:**
1. Create a wide table and observe full-table scans via `input_file_name()`
2. Try to enable clustering → hit the `DELTA_CLUSTERING_COLUMN_MISSING_STATS` error
3. Unblock: extend stats coverage, rewrite files, then cluster
4. Measure the improvement

---


In [75]:
# ============================================================
# 5⃣ SETUP — Create a wide denormalized table
# ============================================================

# Join several LEGO tables into a single wide table (35+ columns).
# The target filter column (theme_name) will land PAST position 32.
# Note: ArcFlow normalizes all column names to snake_case.

# Disable cache to prevent caching from skewing the perf results
spark.conf.set('spark.synapse.vegas.useCache', True)

wide_df = spark.sql(f"""
    SELECT
        -- web_order columns (1-8)
        wo.order_id,
        wo.order_number,
        wo.order_date,
        wo.customer_id          AS wo_customer_id,
        wo.shipping_country,
        wo.source               AS order_source,
        wo.order_total,
        wo.generated_at         AS order_generated_at,

        -- web_order_line columns (9-15)
        wol.line_number,
        wol.set_num,
        wol.part_num            AS line_part_num,
        wol.item_name,
        wol.quantity             AS line_quantity,
        wol.unit_price,
        wol.extended_price,

        -- customer columns (16-23)
        c.name                  AS customer_name,
        c.email,
        c.country,
        c.loyalty_tier,
        c.member_since,
        c.preferred_source,
        c.organization_id       AS customer_org_id,
        c.generated_at          AS customer_generated_at,

        -- sets columns (24-29)
        s.name                  AS set_name,
        s.year                  AS set_year,
        s.num_parts,
        s.img_url               AS set_img_url,
        s.theme_id,
        s.generated_at          AS set_generated_at,

        -- set_price_history columns (30-34)
        sph.price,
        sph.currency             AS price_currency,
        sph.effective_from,
        sph.effective_to,
        sph.change_reason,

        -- themes columns (35-37) — PAST THE 32-COLUMN BOUNDARY
        t.name                  AS theme_name,
        t.parent_id             AS theme_parent_id,
        t.generated_at          AS theme_generated_at

    FROM {ORIG_SCHEMA}.web_order_line wol
    JOIN {ORIG_SCHEMA}.web_order wo      ON wol.order_id = wo.order_id
    JOIN {ORIG_SCHEMA}.customer c        ON wo.customer_id = c.customer_id
    LEFT JOIN {ORIG_SCHEMA}.sets s       ON wol.set_num = s.set_num
    LEFT JOIN {ORIG_SCHEMA}.set_price_history sph ON s.set_num = sph.set_num
    LEFT JOIN {ORIG_SCHEMA}.themes t     ON s.theme_id = t.id
""")

num_cols = len(wide_df.columns)
print(f"\U0001f4d0 Wide table has {num_cols} columns")

# Write as an UNCLUSTERED Delta table (no clustering yet!)
WIDE_TABLE = "wide_order_analysis"

spark.sql(f"DROP TABLE IF EXISTS {FAST_SCHEMA}.{WIDE_TABLE}")

wide_df.write \
    .format("delta") \
    .option("delta.targetFileSize", "1m") \
    .saveAsTable(f"{FAST_SCHEMA}.{WIDE_TABLE}")

show_metrics(f"{FAST_SCHEMA}.{WIDE_TABLE}", "unclustered")

# Show column positions
print(f"\n\U0001f4cb Column positions:")
for i, col in enumerate(wide_df.columns, 1):
    marker = " \U0001f448 FILTER COLUMN (past position 32!)" if col == "theme_name" else ""
    boundary = " \u2500\u2500 stats boundary \u2500\u2500" if i == 32 else ""
    if i <= 3 or i >= 30 or col == "theme_name":
        print(f"   {i:>3}. {col}{boundary}{marker}")
    elif i == 4:
        print(f"       ...")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 77, Finished, Available, Finished, False)

📐 Wide table has 37 columns
   bronze_fast.wide_order_analysis (unclustered):       9 files  |      23.4 MB  |  avg   2667.7 KB/file

📋 Column positions:
     1. order_id
     2. order_number
     3. order_date
       ...
    30. price
    31. price_currency
    32. effective_from ── stats boundary ──
    33. effective_to
    34. change_reason
    35. theme_name 👈 FILTER COLUMN (past position 32!)
    36. theme_parent_id
    37. theme_generated_at


In [104]:
# ============================================================
# 5⃣ SETUP — Create a wide denormalized table
# ============================================================

# Join LEGO manufacturing tables into a single wide table (35+ columns).
# manufacturing_event is the high-frequency IoT fact table — the biggest in the lakehouse.
# The target filter column (part_num) will land PAST position 32.
# Note: ArcFlow normalizes all column names to snake_case.

spark.conf.set('spark.synapse.vegas.useCache', True)

wide_df = spark.sql(f"""
    SELECT
        -- manufacturing_event columns (1-10)
        me.event_id,
        me.timestamp             AS event_timestamp,
        me.machine_id,
        me.color_id,
        me.mold_temp,
        me.injection_pressure,
        me.cycle_time_ms,
        me.defect_detected,
        me.defect_type,
        me.batch_id,

        -- production_line columns (11-15)
        pl.plant,
        pl.cell_type,
        pl.capacity              AS line_capacity,
        pl.installed_year,
        pl.last_maintenance,

        -- mold columns (16-24)
        m.mold_id,
        m.cavity_count,
        m.material               AS mold_material,
        m.max_shots,
        m.current_shots,
        m.status                 AS mold_status,
        m.commission_date,
        m.last_resurfaced,
        m.assigned_line_id,

        -- colors columns (25-31)
        c.name                   AS color_name,
        c.rgb,
        c.is_trans,
        c.num_parts              AS color_num_parts,
        c.num_sets               AS color_num_sets,
        c.y_1                     AS color_first_year,
        c.y_2                     AS color_last_year,

        -- parts columns (32-36) — PAST THE 32-COLUMN BOUNDARY
        p.part_cat_id,
        me.part_num,
        p.name                   AS part_name,
        p.part_material,
        pc.name                  AS part_category_name

    FROM {FAST_SCHEMA}.manufacturing_event me
    JOIN {FAST_SCHEMA}.production_line pl  ON me.machine_id = pl.line_id
    LEFT JOIN (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY assigned_line_id, part_num
            ORDER BY CASE status WHEN 'active' THEN 1 ELSE 2 END, current_shots DESC
        ) AS rn
        FROM {FAST_SCHEMA}.mold
    ) m ON m.assigned_line_id = me.machine_id AND m.part_num = me.part_num AND m.rn = 1
    JOIN {FAST_SCHEMA}.colors c            ON me.color_id = c.id
    JOIN {FAST_SCHEMA}.parts p             ON me.part_num = p.part_num
    JOIN {FAST_SCHEMA}.part_categories pc  ON p.part_cat_id = pc.id
""")

num_cols = len(wide_df.columns)
print(f"\U0001f4d0 Wide table has {num_cols} columns")

# Write as an UNCLUSTERED Delta table (no clustering yet!)
WIDE_TABLE = "production_analysis"

spark.sql(f"DROP TABLE IF EXISTS {FAST_SCHEMA}.{WIDE_TABLE}")

# write table, repartition to force many files (don't do this!)
wide_df.repartition(100).write \
    .option("delta.targetFileSize", "1m") \
    .saveAsTable(f"{FAST_SCHEMA}.{WIDE_TABLE}")

show_metrics(f"{FAST_SCHEMA}.{WIDE_TABLE}", "unclustered")

# Show column positions
print(f"\n\U0001f4cb Column positions:")
for i, col in enumerate(wide_df.columns, 1):
    marker = " \U0001f448 FILTER COLUMN (past position 32!)" if col == "part_num" else ""
    boundary = " \u2500\u2500 stats boundary \u2500\u2500" if i == 32 else ""
    if i <= 3 or i >= 30 or col == "part_num":
        print(f"   {i:>3}. {col}{boundary}{marker}")
    elif i == 4:
        print(f"       ...")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 106, Finished, Available, Finished, False)

📐 Wide table has 36 columns
   bronze_fast.production_analysis (unclustered):     100 files  |     484.3 MB  |  avg   4959.3 KB/file

📋 Column positions:
     1. event_id
     2. event_timestamp
     3. machine_id
       ...
    30. color_first_year
    31. color_last_year
    32. part_cat_id ── stats boundary ──
    33. part_num 👈 FILTER COLUMN (past position 32!)
    34. part_name
    35. part_material
    36. part_category_name


In [105]:
# 🧱 Let's explore the wide table — look at the LEGO manufacturing data!
print("🏭 Sample rows from the wide denormalized table\n")
display(spark.sql(f"""
    SELECT part_num, part_name, plant, color_name,
           ROUND(mold_temp, 1) AS mold_temp, defect_detected, batch_id
    FROM {FAST_SCHEMA}.production_analysis
    ORDER BY event_timestamp DESC
    LIMIT 15
"""))

# Show part distribution
print("\n🏷️ Top parts by manufacturing event volume\n")
display(spark.sql(f"""
    SELECT part_num, part_name, COUNT(*) AS events,
           ROUND(AVG(mold_temp), 1) AS avg_temp,
           SUM(CASE WHEN defect_detected THEN 1 ELSE 0 END) AS defects
    FROM {FAST_SCHEMA}.production_analysis
    GROUP BY part_num, part_name
    ORDER BY events DESC
    LIMIT 10
"""))

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 107, Finished, Available, Finished, False)

🏭 Sample rows from the wide denormalized table



SynapseWidget(Synapse.DataFrame, 976a3663-c1cd-4a09-b3bf-132ed5e64357)


🏷️ Top parts by manufacturing event volume



SynapseWidget(Synapse.DataFrame, 7b017063-774b-4e15-97bc-4b120c2e22f8)

In [106]:
# ============================================================
# 5⃣ BENCHMARK — Query filtering on a column past position 32
# ============================================================

# Pick a frequently produced part to filter on
sample_part = spark.sql(f"""
    SELECT part_num, COUNT(*) AS cnt
    FROM {FAST_SCHEMA}.{WIDE_TABLE}
    GROUP BY part_num
    ORDER BY cnt DESC LIMIT 1
""").collect()[0]["part_num"]

print(f"\U0001f50d Filtering on part_num = '{sample_part}'")
print(f"   part_num is at column position {wide_df.columns.index('part_num') + 1} (past the 32-col stats window)\n")

# How many files does the query scan?
total_files = spark.table(f"{FAST_SCHEMA}.{WIDE_TABLE}").selectExpr("input_file_name()").distinct().count()
filtered_files = spark.sql(f"""
    SELECT input_file_name() AS f
    FROM {FAST_SCHEMA}.{WIDE_TABLE}
    WHERE part_num = '{sample_part}'
""").select("f").distinct().count()

print(f"\U0001f4c1 File scan analysis:")
print(f"   Total files in table:       {total_files}")
print(f"   Files read by filtered query: {filtered_files}")
print(f"   Files pruned:               {total_files - filtered_files}")
print(f"   Pruning ratio:              {((total_files - filtered_files) / max(total_files, 1)) * 100:.0f}%")

if filtered_files >= total_files:
    print(f"\n   \u26a0\ufe0f  No files pruned! Every file was scanned despite filtering on part_num.")
    print(f"   Without stats on part_num, Delta cannot skip any files.")

# Baseline timing
with benchmark_op("Data Skipping Stats", "before (no stats past col 32)", spark):
    query_no_stats = spark.sql(f"""
        SELECT *
        FROM {FAST_SCHEMA}.{WIDE_TABLE}
        WHERE part_num = '{sample_part}'
    """)
    display(query_no_stats)

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 108, Finished, Available, Finished, False)

🔍 Filtering on part_num = '4066pr9994'
   part_num is at column position 33 (past the 32-col stats window)

📁 File scan analysis:
   Total files in table:       100
   Files read by filtered query: 100
   Files pruned:               0
   Pruning ratio:              0%

   ⚠️  No files pruned! Every file was scanned despite filtering on part_num.
   Without stats on part_num, Delta cannot skip any files.


SynapseWidget(Synapse.DataFrame, afc3286b-7ddf-44dd-b18d-dfc3eacef1e5)

⏱️  Data Skipping Stats [before (no stats past col 32)]: 3523.71 ms

  ┌──────────────────────────────────────────────────────────┐
  │  Data Skipping Stats                                     │
  ├──────────────────────────────────────────────────────────┤
  │  State                          Time (ms)        Factor  │
  ├──────────────────────────────────────────────────────────┤
  │  before (no stats past col 32)     3523.71      baseline  │
  │  after (stats + clustering)        404.28   8.7x faster  │
  └──────────────────────────────────────────────────────────┘


In [107]:
# ============================================================
# 5⃣ DIAGNOSE — Try to enable clustering on theme_name
# ============================================================

# You might think: "just cluster on theme_name to co-locate the data!"
# But clustering REQUIRES stats on the clustering column.
# Since theme_name is at position 35 (past the 32-col stats boundary),
# Delta will REJECT the clustering request.


print("\U0001f9ea Attempting: ALTER TABLE ... CLUSTER BY (part_num)\n")

try:
    spark.sql(f"ALTER TABLE {FAST_SCHEMA}.{WIDE_TABLE} CLUSTER BY (part_num)")
    print("\u2705 Clustering enabled (stats already exist for part_num)")
except Exception as e:
    error_msg = str(e)
    print(f"\u274c BLOCKED! Spark error:\n")
    # Extract the key error message
    if "DELTA_CLUSTERING_COLUMN_MISSING_STATS" in error_msg:
        print(f"   DELTA_CLUSTERING_COLUMN_MISSING_STATS")
        print(f"   Clustering column 'theme_name' doesn't have statistics collected.")
        print(f"\n\U0001f4a1 Why? Delta only collects min/max stats for the first 32 columns.")
        print(f"   part_num is at position {wide_df.columns.index('part_num') + 1} \u2014 outside the stats window.")
        print(f"   Without stats, clustering can't determine how to organize files.")
    else:
        print(f"   {error_msg[:500]}")

print(f"\n\U0001f4ca Current data skipping configuration:")
props = spark.sql(f"SHOW TBLPROPERTIES {FAST_SCHEMA}.{WIDE_TABLE}").collect()
skip_props = {r['key']: r['value'] for r in props if 'skip' in r['key'].lower() or 'indexed' in r['key'].lower()}
if skip_props:
    for k, v in skip_props.items():
        print(f"   {k} = {v}")
else:
    print(f"   delta.dataSkippingNumIndexedCols = 32  (default \u2014 not explicitly set)")

print(f"\n\U0001f4cb Table has {len(wide_df.columns)} columns")
print(f"   Stats collected for:  columns 1-32")
print(f"   Stats MISSING for:   columns 33-{len(wide_df.columns)}")
print(f"   part_num position: {wide_df.columns.index('part_num') + 1} \u2190 not enabled for stats\u2192 clustering blocked")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 109, Finished, Available, Finished, False)

🧪 Attempting: ALTER TABLE ... CLUSTER BY (part_num)

❌ BLOCKED! Spark error:

   DELTA_CLUSTERING_COLUMN_MISSING_STATS
   Clustering column 'theme_name' doesn't have statistics collected.

💡 Why? Delta only collects min/max stats for the first 32 columns.
   part_num is at position 33 — outside the stats window.
   Without stats, clustering can't determine how to organize files.

📊 Current data skipping configuration:
   delta.dataSkippingNumIndexedCols = 32  (default — not explicitly set)

📋 Table has 36 columns
   Stats collected for:  columns 1-32
   Stats MISSING for:   columns 33-36
   part_num position: 33 ← not enabled for stats→ clustering blocked


### 🎯 Challenge: Unblock Clustering on `part_num`

You saw the `DELTA_CLUSTERING_COLUMN_MISSING_STATS` error. Stats are only collected for the first 32 columns, and `part_num` is at position 33.

**Your task:** Fix it in 3 steps:
1. Apply one of the 3 methods to customize columns indexed with stats:
    - `delta.dataSkippingNumIndexedCols` to `-1` (collect stats on ALL columns)
    - `delta.dataSkippingStatsColumns` to `part_num` (collect stats on ALL columns)
    - Move `part_num` within the first 32 columns (`ALTER TABLE ... ALTER COLUMN part_num AFTER ...`)

2. Run `OPTIMIZE FULL` to rewrite files with complete stats
3. Enable `CLUSTER BY (part_num)`

> 💡 You'll need three `spark.sql()` calls with ALTER TABLE, OPTIMIZE FULL, and ALTER TABLE again.

In [108]:
WIDE_TABLE = "production_analysis"

# YOUR CODE HERE
# Step 1: Extend stats coverage to all columns
# spark.sql(f"ALTER TABLE {FAST_SCHEMA}.{WIDE_TABLE} SET TBLPROPERTIES ...")

# Step 2: Now enable clustering
# spark.sql(f"ALTER TABLE ...")

# Step 3: Rewrite files with complete stats
# spark.sql(f"OPTIMIZE ... FULL")


StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 110, Finished, Available, Finished, False)

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,numFilesUpdatedWithoutRewrite:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesUpdatedWithoutRewrite:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemovedBreakdown:array<struct<reason:string,metrics:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>>>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,

<details>
  <summary><strong>🔑 Solution:</strong> Click to reveal</summary>

<br/>

```python
WIDE_TABLE = "production_analysis"

# Step 1: Tell Delta to collect stats on ALL columns
spark.sql(f"ALTER TABLE {FAST_SCHEMA}.{WIDE_TABLE} SET TBLPROPERTIES ('delta.dataSkippingNumIndexedCols' = '-1')")

# Step 2: Enable clustering
spark.sql(f"ALTER TABLE {FAST_SCHEMA}.{WIDE_TABLE} CLUSTER BY (part_num)")

# Step 3: Rewrite existing files to include the new stats
spark.sql(f"OPTIMIZE {FAST_SCHEMA}.{WIDE_TABLE} FULL")
```

**Why the order matters:**
1. Changing the property only affects _future_ file writes
2. `OPTIMIZE FULL` rewrites _all_ existing files with the new stats config
3. Only THEN can clustering be enabled — it requires stats on the clustering column

</details>

---

In [109]:
# ============================================================
# 5⃣ RE-BENCHMARK — Same query, now with stats + clustering
# ============================================================

WIDE_TABLE = "production_analysis"

print(f"\U0001f680 Running same query: WHERE part_num = '{sample_part}'\n")

# Check file pruning again
total_files = spark.table(f"{FAST_SCHEMA}.{WIDE_TABLE}").selectExpr("input_file_name()").distinct().count()
filtered_files = spark.sql(f"""
    SELECT input_file_name() AS f
    FROM {FAST_SCHEMA}.{WIDE_TABLE}
    WHERE part_num = '{sample_part}'
""").select("f").distinct().count()

print(f"\U0001f4c1 File scan analysis (after fix):")
print(f"   Total files in table:       {total_files}")
print(f"   Files read by filtered query: {filtered_files}")
print(f"   Files pruned:               {total_files - filtered_files}")
print(f"   Pruning ratio:              {((total_files - filtered_files) / max(total_files, 1)) * 100:.0f}%")

if filtered_files < total_files:
    print(f"\n   \u2705 Data skipping is working! Most files were pruned.")

# Timing comparison
with benchmark_op("Data Skipping Stats", "after (stats + clustering)", spark):
    query_no_stats = spark.sql(f"""
        SELECT *
        FROM {FAST_SCHEMA}.{WIDE_TABLE}
        WHERE part_num = '{sample_part}'
    """)
    display(query_no_stats)

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 111, Finished, Available, Finished, False)

🚀 Running same query: WHERE part_num = '4066pr9994'

📁 File scan analysis (after fix):
   Total files in table:       484
   Files read by filtered query: 1
   Files pruned:               483
   Pruning ratio:              100%

   ✅ Data skipping is working! Most files were pruned.


SynapseWidget(Synapse.DataFrame, acaf9207-45ff-4711-9070-a503b3e3fe20)

⏱️  Data Skipping Stats [after (stats + clustering)]: 949.29 ms

  ┌──────────────────────────────────────────────────────────┐
  │  Data Skipping Stats                                     │
  ├──────────────────────────────────────────────────────────┤
  │  State                          Time (ms)        Factor  │
  ├──────────────────────────────────────────────────────────┤
  │  before (no stats past col 32)     3523.71      baseline  │
  │  after (stats + clustering)        949.29   3.7x faster  │
  └──────────────────────────────────────────────────────────┘


### 💡 What Just Happened?

Delta Lake collects **min/max statistics** per column per file. These stats power **data skipping** — the ability to prune entire files from a scan when the filter value falls outside a file’s min/max range.

By default, stats are only collected for the **first 32 columns** (`delta.dataSkippingNumIndexedCols = 32`). This creates two problems for wide tables:

1. **Queries silently scan everything** — no stats means no file pruning, even with good data layout
2. **Clustering is blocked** — `DELTA_CLUSTERING_COLUMN_MISSING_STATS` prevents you from clustering on columns without stats

| Scenario | Stats? | Clustering? | Files pruned | Scan cost |
|----------|--------|-------------|--------------|-----------|
| `theme_name` at position 35, default settings | ❌ No | ❌ Blocked | 0 | Full scan |
| After `dataSkippingNumIndexedCols = -1` + OPTIMIZE FULL + CLUSTER BY | ✅ Yes | ✅ Enabled | Most files | Minimal scan |

**Three ways to fix it:**

| Fix | Table property | When to use |
|-----|---------------|-------------|
| Reorder columns | N/A — schema change | Best if you control the schema; put high-cardinality filter columns first |
| Index all columns | `delta.dataSkippingNumIndexedCols = -1` | Easy fix; small write overhead per column |
| Index specific columns | `delta.dataSkippingStatsColumns = 'col1,col2'` | Surgical; only pay stats cost for columns you filter on |

> ⚠️ **Order matters:** You must (1) extend stats coverage, (2) enable clustering, and THEN (3) `OPTIMIZE FULL` to rewrite files with new stats, Clustering requires columns be _enabled_ for stats collection.

---


---

# 🏆 Summary Dashboard

All five optimizations applied. Here's the full impact across every exercise.

---


In [110]:
# ============================================================
# SUMMARY — All benchmark results
# ============================================================

print("=" * 62)
print("  🏆  PERFORMANCE IMPACT SUMMARY")
print("=" * 62)

for scenario, states in benchmarks.items():
    if isinstance(states, dict):
        baseline_key = next(iter(states))
        baseline_ms = states[baseline_key]
        best_ms = min(states.values())
        W = 58
        print(f"\n  \u250c{'\u2500' * W}\u2510")
        title = f"\033[1m{scenario}\033[0m"
        title_pad = W - 2 - len(scenario)
        print(f"  \u2502  {title}{' ' * title_pad}\u2502")
        print(f"  \u251c{'\u2500' * W}\u2524")
        print(f"  \u2502  {'State':<28}{'Time (ms)':>12}{'Factor':>14}  \u2502")
        print(f"  \u251c{'\u2500' * W}\u2524")
        for s, ms in states.items():
            ratio = baseline_ms / max(ms, 0.001)
            if s == baseline_key:
                visible_tag = "baseline"
                tag = visible_tag
            elif ms <= best_ms:
                visible_tag = f"{ratio:.1f}x faster"
                tag = f"\033[1;32m{visible_tag}\033[0m"
            else:
                visible_tag = f"{ratio:.1f}x"
                tag = f"\033[1;34m{visible_tag}\033[0m"
            pad = 14 - len(visible_tag)
            print(f"  \u2502  {s:<28}{ms:>12.2f}{' ' * pad}{tag}  \u2502")
        print(f"  \u2514{'\u2500' * W}\u2518")

print(f"\n{'=' * 62}")
print("""
KEY TAKEAWAYS
──────────────
1. OPTIMIZE compacts small files — but it's REACTIVE (run after the damage)
2. Optimize Write bin-packs at write time — PROACTIVE, prevents small files at source
3. Liquid clustering enables data skipping — selective queries skip irrelevant files
4. Deletion vectors eliminate write amplification — DELETEs don't rewrite files
5. Data skipping stats must cover filter columns — wide tables need explicit config
""")

StatementMeta(, fcbd4df1-7649-425e-8985-c4447f4c1d00, 112, Finished, Available, Finished, False)

  🏆  PERFORMANCE IMPACT SUMMARY

  ┌──────────────────────────────────────────────────────────┐
  │  Deletion Vectors                                        │
  ├──────────────────────────────────────────────────────────┤
  │  State                          Time (ms)        Factor  │
  ├──────────────────────────────────────────────────────────┤
  │  without DVs (file rewrite)       5257.22      baseline  │
  │  with DVs (sidecar only)          5525.15          1.0x  │
  └──────────────────────────────────────────────────────────┘

  ┌──────────────────────────────────────────────────────────┐
  │  Schema Inference                                        │
  ├──────────────────────────────────────────────────────────┤
  │  State                          Time (ms)        Factor  │
  ├──────────────────────────────────────────────────────────┤
  │  inferred (file scan)            26539.77      baseline  │
  │  static (no scan)                  966.29  27.5x faster  │
  └──────────────────

---

### 📝 What the Optimized Pipeline Configures Automatically

The `seed_lego_delta_tables` Spark Job Definition uses ArcFlow's `SparkConfigurator` to set these best-practice configs at session startup:

| Setting | Value | What it does |
|---------|-------|-------------|
| `autoCompact.enabled` | `true` | Compacts small files after writes |
| `optimizeWrite.enabled` | `true` | Bin-packs data during writes |
| `targetFileSize.adaptive.enabled` | `true` | Adapts target file size to workload |
| `enableDeletionVectors` | `true` | Marks deleted rows instead of rewriting files |
| `optimize.fast.enabled` | `true` | Faster OPTIMIZE via incremental compaction |
| `parquet.compression.codec` | `zstd` | Better compression ratio than snappy |
| `sql.adaptive.enabled` | `true` | Adaptive Query Execution (AQE) |
| `native.enabled` | `true` | Native Execution Engine |

> 💡 **The best optimization is the one you never have to run manually.**

---